In [1]:
#Import models and libraries
from DT.DecisionTree import DecisionTree
from sklearn.tree import DecisionTreeClassifier
from GNB.gaussian_naive_bayes import GNB
from LogisticRegresssion.LogisticRegression import LogReg
from tensorflow import keras
from SVM.linear_svm import LinearSVMScartch
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report , f1_score
from sklearn.model_selection import train_test_split
from Kfolds import run_kfold



In [2]:
#Download Data
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()


In [3]:
y_train_bin = (y_train == 6).astype(int)
y_test_bin  = (y_test == 6).astype(int)

# Flatten
# X_train = X_train.reshape(X_train.shape[0], -1)
# X_test = X_test.reshape(X_test.shape[0], -1)


In [4]:
# Normalize
X_train = X_train / 255.0
X_test = X_test / 255.0

In [5]:
import numpy as np
from skimage.feature import hog

def extract_hog_features(X):
    features = []

    for img in X:
        img_2d = img.reshape(28, 28)

        hog_features = hog(
            img_2d,
            orientations=9,
            pixels_per_cell=(4, 4),
            cells_per_block=(2, 2),
            block_norm='L2-Hys',
            feature_vector=True
        )

        features.append(hog_features)

    return np.array(features)

In [6]:
X_train_HOG = extract_hog_features(X_train)
X_test_HOG = extract_hog_features(X_test)

In [7]:
pca = PCA(n_components=50)  

X_train_hog_pca = pca.fit_transform(X_train_HOG)
X_test_hog_pca = pca.transform(X_test_HOG)

In [8]:
print(X_test_hog_pca.shape)

(10000, 50)


In [9]:
#Validation split from training data
X_train, X_val, y_train, y_val = train_test_split(
    X_train_hog_pca, y_train_bin,
    test_size=0.2,
    stratify=y_train_bin,
    random_state=42
)

In [ ]:
#Decision Tree results
def train_DT(X, y):
    dt = DecisionTree(maxDepth = 12,
    minSamplesSplit = 10,
    minSampleLeafs = 5,
    criterion = "entropy",
    maxFeatures = "sqrt",
    classWeights = {0: 1, 1: 5}
    )
    dt.fit(X, y)
    return dt
def predict_DT(model, X):
    return model.predict(X)

#Validation
print("Validation results")
run_kfold(X_train , y_train , train_DT , predict_DT)

dt = DecisionTree(maxDepth = 12,
    minSamplesSplit = 10,
    minSampleLeafs = 5,
    criterion = "entropy",
    maxFeatures = "sqrt",
    classWeights = {0: 1, 1: 5}
    )


dt.fit(X_train , y_train)
predictions = dt.predict(X_test_hog_pca)
print(classification_report(
    y_test_bin, predictions,
    target_names=["Not 6", "Is 6"]
))



Validation results
Fold 1 → F1: 0.8330
Fold 2 → F1: 0.8894
Fold 3 → F1: 0.8990
Fold 4 → F1: 0.7992
Fold 5 → F1: 0.8659

K-Fold Avg F1: 0.8573089026275591
              precision    recall  f1-score   support

       Not 6       0.99      0.99      0.99      9042
        Is 6       0.87      0.95      0.91       958

    accuracy                           0.98     10000
   macro avg       0.93      0.97      0.95     10000
weighted avg       0.98      0.98      0.98     10000



In [ ]:
# Train
gnb = GNB()
gnb.gaussian_naive_train(X_train, y_train)

# Tune weights
best_weight = None
best_score = -1

for w in [1, 1.5, 2, 3, 4, 5, 7, 10]:
    class_weights = {0: 1.0, 1: w}
    preds = gnb.predict(X_val, class_weights=class_weights)
    score = f1_score(y_val, preds, pos_label=1)

    print(f"Weight {w} → F1: {score:.4f}")

    if score > best_score:
        best_score = score
        best_weight = w

print("\n🔥 Best weight:", best_weight)

def train_gnb(X, y):
    gnb = GNB()
    gnb.gaussian_naive_train(X, y)
    return gnb


def predict_gnb(model, X):
    return model.predict(
        X,
        class_weights={0: 1.0, 1: best_weight}
    )


# ======================
# STEP 2: K-FOLD
# ======================
print("\n=== K-FOLD VALIDATION ===")
run_kfold(X_train, y_train, train_gnb, predict_gnb, k=5)


# ======================
# STEP 3: FINAL TEST
# ======================
print("\n=== FINAL TEST RESULTS ===")

gnb = GNB()
gnb.gaussian_naive_train(X_train, y_train)

predictions = gnb.predict(
    X_test_hog_pca,   # ✅ CORRECT
    class_weights={0: 1.0, 1: best_weight}
)

print(classification_report(
    y_test_bin,
    predictions,
    target_names=["Not 6", "Is 6"]
))

Weight 1 → F1: 0.9309
Weight 1.5 → F1: 0.9392
Weight 2 → F1: 0.9405
Weight 3 → F1: 0.9485
Weight 4 → F1: 0.9504
Weight 5 → F1: 0.9510
Weight 7 → F1: 0.9525
Weight 10 → F1: 0.9536

🔥 Best weight: 10

=== K-FOLD VALIDATION ===
Fold 1 → F1: 0.9472
Fold 2 → F1: 0.9548
Fold 3 → F1: 0.9574
Fold 4 → F1: 0.9525
Fold 5 → F1: 0.9529

K-Fold Avg F1: 0.9529874832956813

=== FINAL TEST RESULTS ===
              precision    recall  f1-score   support

       Not 6       0.99      1.00      1.00      9042
        Is 6       0.98      0.94      0.96       958

    accuracy                           0.99     10000
   macro avg       0.99      0.97      0.98     10000
weighted avg       0.99      0.99      0.99     10000



In [ ]:
#N7ot adham fi pickle
import pickle


model_package = {
    "model": gnb,
    "pca": pca,
    "best_weight": best_weight,
    "hog_params": {
        "orientations": 9,
        "pixels_per_cell": (4, 4),
        "cells_per_block": (2, 2),
        "block_norm": "L2-Hys"
    }
}

# 3. Save to file
with open("pickleGNB/gnb_digit6_package.pkl", "wb") as f:
    pickle.dump(model_package, f)

In [ ]:
lg = LogReg(max_iterations=1000 , learning_rate=0.1 , threshold=0.5)
print("Validation results")
run_kfold(X_val , y_val , lg.fit , lg.predict)

lg.fit(X_train , y_train)
predictions = lg.predict(X_test_hog_pca)
print(classification_report(
    y_test_bin,
    predictions,
    target_names=["Not 6", "Is 6"]
))

              precision    recall  f1-score   support

       Not 6       0.99      1.00      1.00      9042
        Is 6       0.99      0.94      0.96       958

    accuracy                           0.99     10000
   macro avg       0.99      0.97      0.98     10000
weighted avg       0.99      0.99      0.99     10000



In [ ]:
y_train_bin = np.where(y_train == 6, 1, -1)
y_test_bin  = np.where(y_test == 6, 1, -1)

In [ ]:
svm = LinearSVMScartch(
    C=1.0,
    learning_rate=0.0001,
    n_epochs=100,
    batch_size=128,
    use_class_weights=True,
    random_state=42
)
print("Validation results")
run_kfold(X_val , y_val , svm.fit , svm.predict)

svm.fit(X_train , y_train)
predictions = svm.predict(X_test_hog_pca)
print(classification_report(
    y_test_bin,
    predictions,
    target_names=["Not 6", "Is 6"]
))

/Users/amirtamer/CAIE/Sem 6/ML/Project/SVM/linear_svm.py:32: RuntimeWarning: divide by zero encountered in scalar divide
  weight_pos = n_samples / (2.0 * n_pos)


IndexError: index 51012 is out of bounds for axis 0 with size 48000